# TP2 Kafka Spark Audit Logs
Notebook pédagogique fourni avec les TP.


Objectif: analyser des journaux d’accès à grande échelle et détecter des anomalies.

In [ ]:
import pandas as pd
logs = pd.read_csv('../datasets/access_logs.csv', parse_dates=['timestamp'])
logs.head()


In [ ]:
# Version pandas pour les postes sans Spark. Remplacer par SparkSession en salle si disponible.
logs['hour'] = logs['timestamp'].dt.hour
failed = logs.groupby('user_id')['success'].apply(lambda s: (~s).sum()).sort_values(ascending=False)
exports = logs[logs.action=='export'].groupby('user_id')['bytes'].sum().sort_values(ascending=False)
night = logs[(logs.hour < 6) | (logs.hour > 20)]
print('Echecs par utilisateur'); print(failed.head())
print('Exports volumineux'); print(exports.head())
print('Accès nocturnes:', len(night))


In [ ]:
# Score de risque simple: échec + accès nocturne + export + IP externe + MFA échoué.
def risk(row):
    score = 0
    score += 2 if not row['success'] else 0
    score += 2 if (row['hour'] < 6 or row['hour'] > 20) else 0
    score += 3 if row['action'] == 'export' else 0
    score += 3 if str(row['ip']).startswith('196.') else 0
    score += 2 if not row['mfa_passed'] else 0
    return score
logs['risk_score'] = logs.apply(risk, axis=1)
alerts = logs[logs.risk_score >= 6].sort_values('risk_score', ascending=False)
alerts.head(20)


In [ ]:
# Pseudo-code Spark Structured Streaming pour extension Kafka
print('''
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('AccessSecuritySOC').getOrCreate()
raw = spark.readStream.format('kafka') \
  .option('kafka.bootstrap.servers','localhost:9092') \
  .option('subscribe','access_logs').load()
# parse JSON, calculer risk_score, écrire vers MongoDB/HDFS
''')
